In [ ]:
import os
import torch
import pandas as pd
from datasets import Dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    AdamW,
    get_scheduler
)
from sklearn.metrics import accuracy_score, classification_report
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# Paths and Device Setup
base_dir = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
dataset_path = os.path.join(base_dir, "dataset", "fake_reviews_dataset.csv")
model_save_path = os.path.join(base_dir, "models", "TinyLlama", "saved_mistral_model")
result_dir = os.path.join(base_dir, "models", "TinyLlama", "results")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
# MODEL_NAME = "ministral/Ministral-3b-instruct"
# HF_TOKEN = "Replace with your token" 
MAX_LENGTH = 64
BATCH_SIZE = 8
NUM_EPOCHS = 3
LEARNING_RATE = 5e-5
GRADIENT_ACCUMULATION_STEPS = 1

# Load and preprocess dataset
data = pd.read_csv(dataset_path)
data = data[["text", "label"]].dropna()

# Train-validation-test split
train_data, val_data, test_data = torch.utils.data.random_split(
    data.to_dict("records"),
    [int(0.7 * len(data)), int(0.15 * len(data)), len(data) - int(0.7 * len(data)) - int(0.15 * len(data))],
    generator=torch.Generator().manual_seed(42)
)

# Convert to Hugging Face datasets
train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)
test_dataset = Dataset.from_list(test_data)

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_auth_token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, use_auth_token=HF_TOKEN, device_map="auto").to(device)

# Add padding token if missing
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "[PAD]"})
    model.resize_token_embeddings(len(tokenizer))

# Tokenize datasets
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
test_dataset = test_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

train_dataset.set_format("torch")
val_dataset.set_format("torch")
test_dataset.set_format("torch")

# Data collator for padding
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# Create DataLoaders
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=data_collator)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=data_collator)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
num_training_steps = len(train_dataloader) * NUM_EPOCHS
scheduler = get_scheduler(
    "linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

scaler = torch.cuda.amp.GradScaler()

# Training Loop
def train_model(model, train_dataloader, val_dataloader, optimizer, scheduler, num_epochs=NUM_EPOCHS):
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")
        model.train()
        total_train_loss = 0
        progress_bar = tqdm(train_dataloader, desc="Training", unit="batch")

        for step, batch in enumerate(progress_bar):
            batch = {k: v.to(device) for k, v in batch.items()}

            optimizer.zero_grad()

            with torch.cuda.amp.autocast():
                outputs = model(**batch)
                loss = outputs.loss

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_train_loss += loss.item()
            progress_bar.set_postfix(loss=f"{loss.item():.4f}")

        avg_train_loss = total_train_loss / len(train_dataloader)

        # Validation
        model.eval()
        total_val_loss = 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in val_dataloader:
                batch = {k: v.to(device) for k, v in batch.items()}

                with torch.cuda.amp.autocast():
                    outputs = model(**batch)
                    loss = outputs.loss

                total_val_loss += loss.item()
                logits = outputs.logits
                preds = torch.argmax(logits, dim=-1).cpu().numpy()
                labels = batch["labels"].cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels)

        avg_val_loss = total_val_loss / len(val_dataloader)
        val_accuracy = accuracy_score(all_labels, all_preds)

        print(f"Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}")

        # Save model checkpoint
        model_dir = os.path.join(model_save_path, f"epoch_{epoch + 1}")
        os.makedirs(model_dir, exist_ok=True)
        model.save_pretrained(model_dir)
        tokenizer.save_pretrained(model_dir)
        print(f"Model saved to {model_dir}")

train_model(model, train_dataloader, val_dataloader, optimizer, scheduler)

# Test Model
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        logits = model(**batch).logits
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
        labels = batch["labels"].cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels)

# Evaluation Metrics
test_accuracy = accuracy_score(all_labels, all_preds)
report = classification_report(all_labels, all_preds, target_names=["AI-Generated", "Human-Written"])

print(f"\nTest Accuracy: {test_accuracy:.4f}")
print("Classification Report:\n", report)

confusion_matrix = torch.zeros(2, 2)
for t, p in zip(all_labels, all_preds):
    confusion_matrix[t, p] += 1

fig, ax = plt.subplots()
sns.heatmap(confusion_matrix.numpy(), annot=True, fmt=".0f", cmap="Blues", xticklabels=["AI-Generated", "Human-Written"], yticklabels=["AI-Generated", "Human-Written"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.savefig(os.path.join(result_dir, "confusion_matrix.png"))
plt.show()

